In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

os.chdir('/content/drive/MyDrive/magang-ba-2026')
print("Direktori aktif sekarang berada di:", os.getcwd())

Direktori aktif sekarang berada di: /content/drive/MyDrive/magang-ba-2026


**MENGGABUNGKAN RAW DATA**

**Menggabungkan data post**

In [10]:
import os
import pandas as pd

# Pastikan direktori tujuan ada
os.makedirs("data/processed", exist_ok=True)

# 1. GABUNGKAN DATA POSTS (Pilot, B1, B2, B3)
posts_files = [
    "data/raw/post_pilot.csv",
    "data/raw/posts_b1.csv",
    "data/raw/posts_b2.csv",
    "data/raw/posts_b3.csv"
]

posts_list = []
for f in posts_files:
    if os.path.exists(f):
        df_p = pd.read_csv(f)
        posts_list.append(df_p)
        print(f"Memuat: {f} ({len(df_p)} baris)")
    else:
        print(f"File tidak ditemukan: {f}")

if posts_list:
    posts_v0 = pd.concat(posts_list, ignore_index=True).drop_duplicates(subset=["post_id"])
    posts_v0.to_csv("data/processed/posts_v0.csv", index=False)
    print(f"\nBerhasil membuat posts_v0.csv dengan total {len(posts_v0)} unique posts.")


Memuat: data/raw/post_pilot.csv (9 baris)
Memuat: data/raw/posts_b1.csv (19 baris)
Memuat: data/raw/posts_b2.csv (27 baris)
Memuat: data/raw/posts_b3.csv (74 baris)

Berhasil membuat posts_v0.csv dengan total 124 unique posts.


**Menggabungkan data comments**

In [11]:
# 2. GABUNGKAN DATA COMMENTS (Batch Processed yang sudah dibersihkan)
comments_files = [
    "data/processed/comments_batch1_merged.csv",
    "data/processed/comments_batch2_merged.csv",
    "data/processed/comments_batch3_processed.csv"
]

comments_list = []
for f in comments_files:
    if os.path.exists(f):
        df_c = pd.read_csv(f)
        comments_list.append(df_c)
        print(f"Memuat: {f} ({len(df_c)} baris)")
    else:
        print(f"File tidak ditemukan: {f}")

if comments_list:
    comments_v0 = pd.concat(comments_list, ignore_index=True)
    comments_v0.to_csv("data/processed/comments_v0.csv", index=False)
    print(f"\nBerhasil membuat comments_v0.csv dengan total {len(comments_v0)} baris.")

Memuat: data/processed/comments_batch1_merged.csv (464 baris)
Memuat: data/processed/comments_batch2_merged.csv (202 baris)
Memuat: data/processed/comments_batch3_processed.csv (1486 baris)

Berhasil membuat comments_v0.csv dengan total 2152 baris.


**PEMERIKSAAN DATA**

In [16]:
import pandas as pd

# 1. Memuat data master v0
posts = pd.read_csv("data/processed/posts_v0.csv")
comments = pd.read_csv("data/processed/comments_v0.csv")

print(f"=== LAPORAN PEMERIKSAAN KELAYAKAN (v0) ===")
print(f"Total Posts: {len(posts)}")
print(f"Total Comments: {len(comments)}\n")

=== LAPORAN PEMERIKSAAN KELAYAKAN (v0) ===
Total Posts: 124
Total Comments: 2152



In [18]:
# 2. Cek Kolom Wajib (menyesuaikan 'published_at')
required_comment_cols = ["comment_id", "post_id", "text_content", "published_at"]
missing_cols = [col for col in required_comment_cols if col not in comments.columns]
if missing_cols:
    print(f"PERINGATAN: Kolom wajib komentar yang hilang: {missing_cols}")
else:
    print("Semua kolom wajib pada comments lengkap.")

Semua kolom wajib pada comments lengkap.


In [19]:
# 3. Cek ID Unik (Duplicate ID)
dup_comments = comments["comment_id"].duplicated().sum()
dup_posts = posts["post_id"].duplicated().sum()
print(f"• Jumlah duplikat comment_id: {dup_comments}")
print(f"• Jumlah duplikat post_id: {dup_posts}")

• Jumlah duplikat comment_id: 214
• Jumlah duplikat post_id: 0


In [20]:
# 4. Cek Teks Kosong
empty_text = comments["text_content"].isna().sum() + (comments["text_content"].astype(str).str.strip() == "").sum()
print(f"• Jumlah baris teks komentar kosong: {empty_text}")

• Jumlah baris teks komentar kosong: 14


In [21]:
# 5. Cek Komentar Tanpa Unggahan Induk (Orphan Comments)
valid_post_ids = set(posts["post_id"])
orphan_comments = comments[~comments["post_id"].isin(valid_post_ids)]
print(f"• Jumlah komentar tanpa unggahan induk (orphan): {len(orphan_comments)}")

• Jumlah komentar tanpa unggahan induk (orphan): 182


In [22]:
# 6. Cek Format Tanggal (menggunakan kolom 'published_at')
try:
    comments["parsed_date"] = pd.to_datetime(comments["published_at"], errors="coerce")
    invalid_dates = comments["parsed_date"].isna().sum()
    print(f"• Jumlah format tanggal tidak valid: {invalid_dates}")
except Exception as e:
    print(f"• Pengecekan tanggal mendeteksi kendala format: {e}")

print("Pemeriksaan kelayakan otomatis selesai!")

• Jumlah format tanggal tidak valid: 1
Pemeriksaan kelayakan otomatis selesai!


**PEMBERSIHAN DAN PENCATATAN**

In [23]:
import pandas as pd
import numpy as np

# 1. Muat data
posts = pd.read_csv("data/processed/posts_v0.csv")
comments = pd.read_csv("data/processed/comments_v0.csv")

# Inisialisasi kolom alasan eksklusi jika belum ada
comments["alasan_eksklusi"] = "Lolos"

In [24]:
# 2. Aturan Eksklusi 1: Duplikat comment_id
dup_mask = comments["comment_id"].duplicated(keep="first")
comments.loc[dup_mask, "alasan_eksklusi"] = "E1_Duplicate_Comment_ID"

In [25]:
# 3. Aturan Eksklusi 2: Teks kosong atau spasi doang
empty_mask = comments["text_content"].isna() | (comments["text_content"].astype(str).str.strip() == "")
comments.loc[empty_mask & (comments["alasan_eksklusi"] == "Lolos"), "alasan_eksklusi"] = "E3_Empty_Text"

In [26]:
# 4. Aturan Eksklusi 3: Komentar tanpa unggahan induk (Orphan)
valid_post_ids = set(posts["post_id"])
orphan_mask = ~comments["post_id"].isin(valid_post_ids)
comments.loc[orphan_mask & (comments["alasan_eksklusi"] == "Lolos"), "alasan_eksklusi"] = "E4_Orphan_Comment"

In [27]:
# 5. Aturan Eksklusi 4: Tanggal tidak valid
comments["parsed_date"] = pd.to_datetime(comments["published_at"], errors="coerce")
date_invalid_mask = comments["parsed_date"].isna()
comments.loc[date_invalid_mask & (comments["alasan_eksklusi"] == "Lolos"), "alasan_eksklusi"] = "E5_Invalid_Date"

In [28]:
# 6. Hitung rekapitulasi eksklusi
print("=== REKAPITULASI EKSKLUSI KOMENTAR ===")
rekap_eksklusi = comments["alasan_eksklusi"].value_counts()
print(rekap_eksklusi)

=== REKAPITULASI EKSKLUSI KOMENTAR ===
alasan_eksklusi
Lolos                      1777
E1_Duplicate_Comment_ID     214
E4_Orphan_Comment           150
E3_Empty_Text                11
Name: count, dtype: int64


In [29]:
# Simpan data komentar yang sudah diberi label alasan eksklusi ke processed
comments.to_csv("data/processed/comments_v0_audited.csv", index=False)
print("\nFile audit disimpan ke: data/processed/comments_v0_audited.csv")


File audit disimpan ke: data/processed/comments_v0_audited.csv


**MENULIS LAPORAN KUALITAS**

In [30]:
import os

# Pastikan folder docs/ ada
os.makedirs("docs", exist_ok=True)

# Isi konten laporan kualitas data D09 (Tanggal: 19 Oktober 2026)
laporan_kualitas_content = """# Laporan Kualitas Data v0 (Hari 09)
**Tanggal Penyusunan:** 19 Oktober 2026

## 1. Ringkasan Eksekutif Pemrosesan Data
Penggabungan data dari seluruh batch (Pilot, Batch 1, Batch 2, dan Batch 3) menghasilkan basis data awal (*v0*) dengan rincian sebagai berikut:
* **Total Baris Mentah (Raw):** 2,152 komentar dan 124 postingan (*posts*)
* **Total Komentar Lolos (Layak Aktif):** 1,777 komentar
* **Total Komentar Dieksklusi:** 375 komentar

## 2. Rincian Eksklusi Berdasarkan Alasan
Berikut adalah rincian data yang dikeluarkan beserta alasannya:
* **E1_Duplicate_Comment_ID:** 214 baris (dikeluarkan karena duplikasi ID komentar yang identik).
* **E4_Orphan_Comment:** 150 baris (dikeluarkan karena `post_id` komentar tidak ditemukan pada master tabel `posts_v0`).
* **E3_Empty_Text:** 11 baris (dikeluarkan karena teks komentar kosong atau hanya berisi spasi).
* **E2_Duplicate_Text:** Ditangani pada tahap batch sebelumnya menggunakan *hash* SHA-256.

## 3. Evaluasi Target Kuantitatif
* Target minimal kumulatif data layak: **1,000 komentar**.
* Hasil aktual data layak: **1,777 komentar**.
* **Status:** Target tercapai dan terlampaui dengan aman (memenuhi syarat untuk dilanjutkan ke tahap pelabelan dan pemodelan).

## 4. Penanganan & Tindak Lanjut
* Data yang lolos disimpan ke `data/processed/comments_v0_audited.csv` dan `comments_v0.csv`.
* File data mentah (`data/raw/`) tetap dijaga keasliannya tanpa ada modifikasi (*frozen*).
"""

# Tulis ke file docs/D09_laporan_kualitas.md
output_path = "docs/D09_laporan_kualitas.md"
with open(output_path, "w", encoding="utf-8") as f:
    f.write(laporan_kualitas_content)

print(f"Berhasil membuat laporan kualitas di: {output_path}")

Berhasil membuat laporan kualitas di: docs/D09_laporan_kualitas.md


**MENULIS DATA DICTIONARY**

In [31]:
import os

# Pastikan folder docs/ ada
os.makedirs("docs", exist_ok=True)

# Isi konten Data Dictionary v0
data_dictionary_content = """# Data Dictionary v0 (Hari 09)
**Tanggal Penyusunan:** 19 Oktober 2026

Dokumen ini mendefinisikan struktur kolom untuk tabel master `posts_v0` dan `comments_v0` yang digunakan dalam sistem analisis bantuan sosial dan pemrosesan komentar.

---

## 1. Tabel `posts_v0` (Unggahan)
Tabel ini menyimpan metadata dari setiap unggahan (*post*) yang menjadi induk dari komentar-komentar terkumpul.

| Nama Kolom | Tipe Data | Wajib (Required) | Arti / Deskripsi | Nilai yang Diizinkan | Contoh Format |
| :--- | :--- | :--- | :--- | :--- | :--- |
| `post_id` | String / Int | Ya | Identifikasi unik untuk setiap unggahan | Unique Identifier | `p_001`, `109283` |
| `platform` | String | Ya | Platform asal unggahan media sosial | `Instagram`, `TikTok`, dll. | `Instagram` |
| `published_at` | Datetime | Ya | Waktu unggahan dipublikasikan | Format ISO 8601 | `2026-10-19 08:30:00` |
| `text_content` | Text | Tidak | Teks deskripsi atau *caption* dari unggahan | Bebas / Teks | "Bantuan sosial tahap 2..." |

---

## 2. Tabel `comments_v0` (Komentar)
Tabel ini menyimpan data interaksi komentar dari para pengguna pada unggahan terkait.

| Nama Kolom | Tipe Data | Wajib (Required) | Arti / Deskripsi | Nilai yang Diizinkan | Contoh Format |
| :--- | :--- | :--- | :--- | :--- | :--- |
| `comment_id` | String / Int | Ya | Identifikasi unik untuk setiap komentar | Unique Identifier | `c_001`, `987654` |
| `post_id` | String / Int | Ya | ID unggahan induk tempat komentar berada | Harus ada di `posts_v0` | `p_001`, `109283` |
| `text_content` | Text | Ya | Isi teks asli dari komentar pengguna | Bebas / Teks | "Bagaimana cara daftar bansos?" |
| `text_original` | Text | Ya | Salinan teks asli sebelum normalisasi | Bebas / Teks | "Bagaimana cara daftar bansos?" |
| `text_hash` | String | Ya | Nilai hash SHA-256 dari teks ternormalisasi | Kode Hash 64 karakter | `e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855` |
| `published_at` | Datetime | Ya | Waktu komentar dipublikasikan | Format ISO 8601 | `2026-10-19 09:15:00` |
| `status_eksklusi` | String | Ya | Status kelayakan data komentar | `Aktif`, `E1`, `E3`, `E4`, `E5` | `Aktif` |
| `alasan_eksklusi` | String | Ya | Keterangan alasan jika data dikeluarkan | `Lolos`, nama kode eksklusi | `Lolos` |
"""

# Tulis ke file docs/data_dictionary_v0.md
output_dd = "docs/data_dictionary_v0.md"
with open(output_dd, "w", encoding="utf-8") as f:
    f.write(data_dictionary_content)

print(f"Berhasil membuat Data Dictionary di: {output_dd}")

Berhasil membuat Data Dictionary di: docs/data_dictionary_v0.md


**RANCANG KOMENTAR SPLIT**

In [36]:
import os
import json
import numpy as np
import pandas as pd

# Pastikan folder target ada
os.makedirs("data/splits", exist_ok=True)

# 1. Muat data posts dan comments aktif
posts = pd.read_csv("data/processed/posts_v0.csv")
comments = pd.read_csv("data/processed/comments_v0_audited.csv")
comments_active = comments[comments["alasan_eksklusi"] == "Lolos"].copy()

In [38]:
# 2. Menerapkan skrip persis seperti di buku panduan
rng = np.random.default_rng(2026)                    # seed dicatat di split_info
plan = {}

for plat, g in posts.groupby("platform"):
    ids = rng.permutation(g.post_id.values)
    n = len(ids); nt = round(n * 0.2); nv = round(n * 0.2)
    plan.update({p: "test" for p in ids[:nt]})
    plan.update({p: "validation" for p in ids[nt:nt + nv]})
    plan.update({p: "train" for p in ids[nt + nv:]})

comments_active["split"] = comments_active.post_id.map(plan)
print("--- DISTRIBUSI SPLIT ---")
print(comments_active.groupby("split").size())

--- DISTRIBUSI SPLIT ---
split
test          521
train         874
validation    382
dtype: int64


In [40]:
# 3. Simpan kembali manifest dan split_info yang diperbarui
manifest_path = "data/splits/manifest_v0.csv"
comments_active[["comment_id", "post_id", "split"]].to_csv(manifest_path, index=False)

split_counts = comments_active["split"].value_counts().to_dict()
split_info = {
    "method": "group_split_by_post_stratified_platform",
    "seed": 2026,
    "date": "2026-10-19",
    "ratios": {"train": 0.6, "validation": 0.2, "test": 0.2},
    "jumlah_komentar_per_split": split_counts
}

split_info_path = "data/splits/split_info_v0.json"
with open(split_info_path, "w", encoding="utf-8") as f:
    json.dump(split_info, f, indent=4)

print("\nManifest dan split info berhasil diperbarui!")


Manifest dan split info berhasil diperbarui!
